# Lesson 12: Multi-Agent Arithmetic — Supervisor + Worker Pattern

## Objective
Build a **Supervisor + Worker** multi-agent system where the Supervisor (LLM) reads the math question, decides which Worker agent to call, and routes to the right specialist.

## Limitation of Lesson 11
- ❌ Routing was based on a hardcoded `operation` string
- ❌ No intelligence — a human had to pre-parse the request
- ❌ Single graph with all logic in one place

## What is NEW in Lesson 12?
- ✅ **Supervisor agent**: an LLM that reads the user's math request and decides routing
- ✅ **Worker agents**: specialist subgraphs (add_worker, multiply_worker, divide_worker)
- ✅ **Supervisor-worker handoff**: Supervisor sets `next` field, which routes to a worker
- ✅ Multi-agent architectural pattern

## Theory: Supervisor-Worker Pattern

```
User: "what is 7 times 8?"

              ┌── add_worker ────→ "adds numbers"
Supervisor ───┼── multiply_worker → "multiplies numbers"
              └── divide_worker ──→ "divides numbers"
```

The **Supervisor**:
- Receives the user's natural language request
- Uses the LLM to classify the operation
- Sets `state["next"]` to route to the right worker

The **Worker**:
- Receives control from the supervisor
- Performs its specialized arithmetic
- Returns result back to the supervisor (or directly to END)

This pattern scales to complex multi-agent systems:
- Each worker has its own LLM, tools, and memory
- The supervisor orchestrates without knowing worker internals


In [ ]:
# ── Cell 1: Imports + LLM Setup ──────────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, Literal, Annotated
import operator
from IPython.display import Image, display

load_dotenv()

# ── Why each line exists ─────────────────────────────────────────────────────
# load_dotenv()    → loads PROJECT_ID and LOCATION from .env file
# vertexai.init()  → authenticates with Google Cloud and sets project/region context
# ChatVertexAI     → LangChain wrapper for Gemini; provides .invoke() interface
# temperature=0    → deterministic output (same question → same routing decision)

vertexai.init(
    project=os.getenv("PROJECT_ID"),
    location=os.getenv("LOCATION")
)

llm = ChatVertexAI(
    model="gemini-2.5-flash",
    temperature=0    # Deterministic: critical for routing decisions
)

print("✓ Vertex AI LLM initialized")
print(f"  Project: {os.getenv('PROJECT_ID')}")
print(f"  Location: {os.getenv('LOCATION')}")
print(f"  Model: gemini-2.5-flash, temperature=0")


In [ ]:
# ── Cell 2: State Schema ────────────────────────────────────────────────────
class State(TypedDict):
    user_request: str           # Natural language math request
    a: Optional[float]          # Extracted first number
    b: Optional[float]          # Extracted second number
    operation: Optional[str]    # Extracted operation
    next: Optional[str]         # Supervisor routing decision
    result: Optional[float]     # Computed result
    explanation: str            # How the result was obtained
    history: Annotated[list, operator.add]

print("✓ Multi-agent state defined")
print("  'next' field drives supervisor → worker routing")


In [ ]:
# ── Cell 3: Supervisor Agent ─────────────────────────────────────────────────
# The Supervisor uses the LLM to:
# 1. Parse the natural language request
# 2. Extract a, b, and operation
# 3. Set state["next"] to route to the right worker

SUPERVISOR_PROMPT = """You are a math request classifier. Given a math question, extract:
- operation: exactly one of "add", "multiply", "divide"
- a: first number (float)
- b: second number (float)

Respond with EXACTLY this format (no other text):
OPERATION: <add|multiply|divide>
A: <number>
B: <number>

Examples:
Question: "what is 7 plus 3?"    → OPERATION: add\nA: 7\nB: 3
Question: "multiply 4 and 5"     → OPERATION: multiply\nA: 4\nB: 5
Question: "divide 20 by 4"       → OPERATION: divide\nA: 20\nB: 4
"""

def supervisor_agent(state: State) -> dict:
    print(f"  [supervisor] Request: '{state['user_request']}'")
    
    messages = [
        SystemMessage(content=SUPERVISOR_PROMPT),
        HumanMessage(content=f"Question: \"{state['user_request']}\"")
    ]
    
    response = llm.invoke(messages)
    raw = response.content.strip()
    print(f"  [supervisor] LLM parsed: {raw.replace(chr(10), ' | ')}")
    
    # Parse LLM response
    lines = {line.split(":")[0].strip(): line.split(":",1)[1].strip() 
             for line in raw.split("\n") if ":" in line}
    
    op = lines.get("OPERATION", "add").lower()
    a  = float(lines.get("A", "0"))
    b  = float(lines.get("B", "0"))
    
    # Set 'next' to route to the appropriate worker
    next_worker = f"{op}_worker"
    print(f"  [supervisor] Routing to: {next_worker} (a={a}, b={b})")
    
    entry = f"Supervisor parsed: '{state['user_request']}' → {op}({a}, {b})"
    return {
        "a": a, "b": b, "operation": op,
        "next": next_worker,
        "history": [entry]
    }

print("✓ Supervisor agent defined")


In [ ]:
# ── Cell 4: Worker Agents ────────────────────────────────────────────────────
# Each worker is a specialized node for ONE operation
# Workers are simple — all the intelligence is in the Supervisor

def add_worker(state: State) -> dict:
    res = state["a"] + state["b"]
    explanation = f"{state['a']} + {state['b']} = {res}"
    print(f"  [add_worker]      {explanation}")
    return {"result": res, "explanation": explanation, "history": [f"Worker: {explanation}"]}

def multiply_worker(state: State) -> dict:
    res = state["a"] * state["b"]
    explanation = f"{state['a']} × {state['b']} = {res}"
    print(f"  [multiply_worker] {explanation}")
    return {"result": res, "explanation": explanation, "history": [f"Worker: {explanation}"]}

def divide_worker(state: State) -> dict:
    if state["b"] == 0:
        return {"result": None, "explanation": "Division by zero", "history": ["Worker: div/0 error"]}
    res = state["a"] / state["b"]
    explanation = f"{state['a']} ÷ {state['b']} = {res:.4f}"
    print(f"  [divide_worker]   {explanation}")
    return {"result": res, "explanation": explanation, "history": [f"Worker: {explanation}"]}

print("✓ Three worker agents defined (add, multiply, divide)")


In [ ]:
# ── Cell 5: Supervisor Router Function ───────────────────────────────────────
# After the supervisor runs, this reads state["next"] to pick a worker

def supervisor_router(state: State) -> Literal["add_worker", "multiply_worker", "divide_worker"]:
    return state["next"]  # "add_worker", "multiply_worker", or "divide_worker"

print("✓ Supervisor router defined — reads state['next']")


In [ ]:
# ── Cell 6: Build Multi-Agent Graph ──────────────────────────────────────────
builder = StateGraph(State)

# Register supervisor and all workers
builder.add_node("supervisor",    supervisor_agent)
builder.add_node("add_worker",    add_worker)
builder.add_node("multiply_worker", multiply_worker)
builder.add_node("divide_worker", divide_worker)

# Execution flow
builder.add_edge(START, "supervisor")

# Supervisor routes to ONE worker based on state["next"]
builder.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "add_worker":      "add_worker",
        "multiply_worker": "multiply_worker",
        "divide_worker":   "divide_worker",
    }
)

# All workers exit to END
builder.add_edge("add_worker",      END)
builder.add_edge("multiply_worker", END)
builder.add_edge("divide_worker",   END)

graph = builder.compile()
print("✓ Multi-agent graph compiled")


In [ ]:
# ── Cell 7: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 8: Test with Natural Language Questions ─────────────────────────────
def run(question: str):
    state = {
        "user_request": question,
        "a": None, "b": None, "operation": None,
        "next": None, "result": None,
        "explanation": "", "history": []
    }
    print(f"\nQuestion: '{question}'")
    print("-" * 50)
    out = graph.invoke(state)
    print(f"Answer: {out['explanation']}")
    return out

out1 = run("what is 7 plus 3?")


In [ ]:
out2 = run("multiply 6 and 9")


In [ ]:
out3 = run("divide 100 by 4")


In [ ]:
out4 = run("what is three times eight?")


## Architecture Deep Dive

```
User Request (natural language)
        ↓
   Supervisor (LLM)
   - Parses question
   - Extracts a, b, operation
   - Sets state["next"] = "xxx_worker"
        ↓
   Router reads state["next"]
        ↓
   ┌─────────────────────────────┐
   │  add_worker / multiply /    │
   │  divide_worker              │
   │  (pure arithmetic, no LLM)  │
   └─────────────────────────────┘
        ↓
      Result
```

### Supervisor Prompt Engineering Tips
- Give clear format instructions (OPERATION: / A: / B:)
- Include examples (few-shot prompting)
- Use `temperature=0` for deterministic parsing
- Add error handling for malformed LLM responses

## Summary

| | Lesson 11 | Lesson 12 |
|--|-----------|-----------|
| Routing | Hardcoded string field | LLM-driven (natural language) |
| Architecture | Single graph | Supervisor + Worker agents |
| Intelligence | None | LLM parses user intent |

## Limitations → What Lesson 13 Solves
- ❌ Each call is stateless — the agent forgets previous calculations
- ❌ "What's double the last result?" would fail — no memory
- ❌ **Lesson 13**: Short-term memory using `MessagesState` to remember conversation history
